In [1]:
import pandas as pd
import duckdb
import os
import numpy as np

con = duckdb.connect("research.duckdb")

#### Get list of every trading day for our history

In [2]:
import QuantLib as ql

firstDate = ql.Date(1, 3, 2014)
secondDate = ql.Date(1, 3, 2026)

calendar2 = ql.UnitedStates(ql.UnitedStates.NYSE)
start_date = ql.Date(min(firstDate.serialNumber(), secondDate.serialNumber()))
end_date = ql.Date(max(firstDate.serialNumber(), secondDate.serialNumber()))

trading_dates_nyse = []
d = start_date
while d <= end_date:
    if calendar2.isBusinessDay(d):
        trading_dates_nyse.append(d)
    d = d + 1

trading_days = ([dt.ISO() for dt in trading_dates_nyse])

trading_days_df = pd.DataFrame({"Date": trading_days})
trading_days_df.head()

,Date
0,2014-03-03
1,2014-03-04
2,2014-03-05
3,2014-03-06
4,2014-03-07


#### Cross join the tradings days and the stock universe and add the CIQ formulas

In [ ]:
def cross_join_add_formulas():
    '''
    This takes the table with all of the tickers and all of the dates
    and performs a cross join    
    '''
    con.execute("DROP TABLE IF EXISTS historical_price_formula")
    con.execute(
        """
    CREATE OR REPLACE TABLE historical_price_formula AS
    WITH
    unique_dates AS (
        SELECT DISTINCT strptime(Date, '%Y-%m-%d')::DATE AS AS_OF 
        FROM trading_days_df
        -- CRITICAL: Remove nulls and unparseable dates
        WHERE Date IS NOT NULL 
        AND Date != 'NaT'
        AND Date != ''
    ),
    unique_assets AS (
        SELECT DISTINCT company_name, IQID 
        FROM dates_w_universe
        -- Optional: Ensure you don't have blank company names
        WHERE company_name IS NOT NULL
    )

    SELECT
        d.AS_OF,
        a.company_name,
        a.IQID,
        CONCAT('=@SPG(', a.IQID, ', \"SP_PRICE_CLOSE_ADJ\", \"', d.AS_OF, '\")') AS adj_close,
        CONCAT('=@SPG(', a.IQID, ', \"SP_VOLUME\", \"', d.AS_OF, '\")') AS volume
    FROM unique_dates d
    CROSS JOIN unique_assets a
    ORDER BY a.company_name, d.AS_OF

    """
    )




In [ ]:
# con.execute("COPY (SELECT * FROM historical_price_ciq_formula LIMIT 500000) TO 'test_price.csv' (HEADER)")

def create_excel_sheets():
    '''
    splits the whole cross joined database into 5000 row sections
    this seemed to be the limit for pulling historical data on excel
    '''
    count = con.execute("SELECT COUNT() FROM historical_price_ciq_formula").df().iloc[0,0]
    batch_size = 100000
    n_batches = (count // batch_size) + 1
    for i in range(n_batches):
        con.execute(f"COPY (SELECT * FROM historical_price_ciq_formula LIMIT {batch_size} OFFSET {i*batch_size}) TO 'datsets/prices/price_formulas/price_formula_{i}.csv' (HEADER)")

# create_excel_sheets()

#### Read the data downloaded from CIQ to a table

In [ ]:
def convert_excel_to_table():
    con.execute("""
        CREATE OR REPLACE TABLE master_historical_prices AS
        SELECT * FROM read_csv(
            'datsets/prices/price_loaded/*.csv',
            header = true,
            quote = '"',
            sample_size = -1,
            ignore_errors = true
        )
    """)

In [29]:
con.execute("SELECT * FROM master_historical_prices  LIMIT 10").df()

,AS_OF,company_name,IQID,adj_close,volume
0,2014-03-03,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.31,"240,741"
1,2014-03-04,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.46,"381,760"
2,2014-03-05,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.43,"180,992"
3,2014-03-06,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.41,"222,328"
4,2014-03-07,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.41,"271,196"
5,2014-03-10,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.46,"111,430"
6,2014-03-11,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.47,"192,889"
7,2014-03-12,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.46,"109,687"
8,2014-03-13,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.36,"111,463"
9,2014-03-14,"1-800-FLOWERS.COM, Inc. (NASDAQGS:FLWS)",4122591,5.42,"67,909"


In [6]:
def check_errors():
  con.execute("""
      SELECT DISTINCT IQID, adj_close
      FROM master_historical_prices
      WHERE TRY_CAST(replace(adj_close, ',', '') AS FLOAT) IS NULL 
        AND adj_close IS NOT NULL
        AND adj_close NOT IN ('NA', 'N/A', '', 'NaN')
  """)


In [ ]:
# change datatype in columns

con.execute("""
            CREATE OR REPLACE TABLE master_historical_prices AS 
            SELECT * REPLACE (
                TRY_CAST(REPLACE(adj_close, ',', '') AS FLOAT) AS adj_close,
                TRY_CAST(REPLACE(volume, ',', '') AS FLOAT) AS volume
            )
            FROM master_historical_prices
""")